# 16 · Market Basket Automated Tests & FastAPI Contract Checks

Run All in VS Code. Covers Apriori metrics, changing cart contents, relevant recommendations, unseen cart abstention, unknown SKUs, **project isolation**, transaction import validation, and live ASGI endpoints. All test data is synthetic.

In [1]:
%run 07_fastapi_integration.ipynb

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Document ingestion functions ready
Offline hybrid retrieval ready
Optional native pgvector adapter defined; credentials/database not required for local demo
Evidence-safe recommender ready
Research adapters and financial metrics ready
Batch research task runner ready
FastAPI routes: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/api/v1/projects/{project_id}/documents', '/api/v1/projects/{project_id}/documents', '/api/v1/projects/{project_id}/knowledge/search', '/api/v1/projects/{project_id}/recommendations/rank']
Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Apriori basket recommender ready; call seed_market_basket_demo() to load synthetic transactions.
Market basket REST routes registered in Group 2 FastAPI app


In [2]:
from math import isclose
from tempfile import TemporaryDirectory
from pprint import pprint
import csv

seed_market_basket_demo(PROJECT_ID,replace=True)
inv=basket_inventory(PROJECT_ID)
assert inv['transaction_count']>100 and inv['is_synthetic']
assert len(inv['products'])>=10
baskets=get_baskets(PROJECT_ID)
counts=apriori_frequent_itemsets(baskets,min_support=.05,max_itemset_size=4)
assert counts[frozenset(['BOSCH-D7050'])]>0
rules=association_rules(counts,len(baskets),min_confidence=.4,min_lift=1.05)
assert len(rules)>0
for r in rules:
    assert 0<r['support']<=1 and 0<r['confidence']<=1 and r['lift']>=1.05
    assert r['joint_transactions']<=r['antecedent_transactions']<=len(baskets)
    assert isclose(r['confidence'],r['joint_transactions']/r['antecedent_transactions'],abs_tol=1e-6)

empty=market_basket_recommend(PROJECT_ID,[])
single=market_basket_recommend(PROJECT_ID,['BOSCH-D7050'])
pair=market_basket_recommend(PROJECT_ID,['BOSCH-D7050','DSC-PG9914'])
unseen=market_basket_recommend(PROJECT_ID,['BOSCH-D7050','AXIS-M3046'])
unknown=market_basket_recommend(PROJECT_ID,['UNKNOWN-SKU'])
assert empty['reason']=='empty_cart'
assert not single['abstained'] and single['recommendations']
assert not pair['abstained'] and pair['recommendations']
assert single['recommendations']!=pair['recommendations']
assert unseen['reason']=='no_historical_cart_pattern' and not unseen['recommendations']
assert unknown['reason']=='unknown_sku'
assert all(x['sku'] not in pair['cart'] for x in pair['recommendations'])
assert basket_inventory('88888888-8888-4888-8888-888888888888')['transaction_count']==0
assert market_basket_recommend('88888888-8888-4888-8888-888888888888',['BOSCH-D7050'])['reason']=='unknown_sku'
try:
    import_basket_rows(PROJECT_ID,[{'transaction_id':'x','sku':'y','product_name':'Z','quantity':0}])
    raise AssertionError('Invalid quantity accepted')
except ValueError:pass
print('PASS: Apriori, metrics, cart recomputation, validation, safe abstention, project isolation')

PASS: Apriori, metrics, cart recomputation, validation, safe abstention, project isolation


In [3]:
import os,asyncio
from httpx import ASGITransport,AsyncClient
os.environ['GROUP2_DEV_MODE']='true'
async def check_basket_api():
    async with AsyncClient(transport=ASGITransport(app=app),base_url='http://testserver') as client:
        base=f'/api/v1/projects/{PROJECT_ID}/market-basket'
        resp=await client.get(base+'/products')
        assert resp.status_code==200 and resp.json()['transaction_count']>0
        resp=await client.post(base+'/recommend',json={'cart':['BOSCH-D7050'],'top_k':3})
        assert resp.status_code==200 and resp.json()['recommendations']
        resp=await client.post(base+'/recommend',json={'cart':['BOSCH-D7050','AXIS-M3046']})
        assert resp.status_code==200 and resp.json()['abstained']
        resp=await client.post(base+'/recommend',json={'cart':['BOSCH-D7050'],'min_support':-1})
        assert resp.status_code==422
        other='88888888-8888-4888-8888-888888888888'
        resp=await client.get(f'/api/v1/projects/{other}/market-basket/products')
        assert resp.status_code==200 and resp.json()['transaction_count']==0
        resp=await client.post(f'/api/v1/projects/{other}/market-basket/recommend',json={'cart':['BOSCH-D7050']})
        assert resp.status_code==200 and resp.json()['abstained']
        resp=await client.post(base+'/transactions/import',json={'rows':[{'transaction_id':'single-test','sku':'INJECTION','product_name':'Test'}],'replace':False})
        assert resp.status_code==200 and resp.json()['transaction_count']==1
        # Never import fake data into other project from path; route scopes all transaction rows.
        return True
assert await check_basket_api()
print('PASS: Market basket FastAPI contracts and project scoping')

PASS: Market basket FastAPI contracts and project scoping
